In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: ฟังก์ชัน Qiskit โดย ColibriTD
> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่พร้อมใช้สำหรับผู้ใช้แผน IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง
## ภาพรวม
ตัวแก้สมการอนุพันธ์ย่อย (PDE) ที่นำเสนอนี้เป็นส่วนหนึ่งของแพลตฟอร์ม Quantum Innovative Computing Kit (QUICK) ของเรา (QUICK-PDE) และบรรจุเป็น Qiskit Function ด้วย QUICK-PDE คุณแก้สมการอนุพันธ์ย่อยเฉพาะโดเมนบน IBM Quantum QPUs ได้ ฟังก์ชันนี้อิงจากอัลกอริทึมที่อธิบายไว้ใน [บทความอธิบาย H-DES ของ ColibriTD](https://arxiv.org/abs/2410.01130) อัลกอริทึมนี้แก้ปัญหา multi-physics ที่ซับซ้อนได้ เริ่มต้นด้วย Computational Fluid Dynamics (CFD) และ Materials Deformation (MD) และกรณีการใช้งานอื่น ๆ กำลังจะตามมา

เพื่อจัดการกับสมการอนุพันธ์ โซลูชันทดลองถูก encode เป็นผลรวมเชิงเส้นของฟังก์ชันตั้งฉาก (โดยทั่วไปคือพหุนาม Chebyshev และโดยเฉพาะ $2^n$ ของพวกมัน โดยที่ $n$ คือจำนวน qubits ที่ encode ฟังก์ชันของคุณ) ซึ่งถูกกำหนดพารามิเตอร์ด้วยมุมของ Variable Quantum Circuit (VQC) ansatz สร้าง state ที่ encode ฟังก์ชัน ซึ่งถูกประเมินโดย observable ที่การผสมผสานช่วยให้ประเมินฟังก์ชันที่ทุกจุดได้ จากนั้นประเมิน loss function ที่สมการอนุพันธ์ถูก encode ไว้ และปรับแต่งมุมในการวนซ้ำแบบ hybrid ดังที่แสดงต่อไปนี้ โซลูชันทดลองค่อย ๆ เข้าใกล้โซลูชันจริงจนกว่าจะได้ผลลัพธ์ที่น่าพอใจ

![Workflow ของฟังก์ชัน QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

นอกจาก hybrid loop นี้ คุณยังต่อ optimizer หลายตัวเข้าด้วยกันได้ ซึ่งมีประโยชน์เมื่อต้องการ optimizer แบบ global เพื่อหาชุดมุมที่ดี แล้วตามด้วย optimizer ที่ละเอียดกว่าเพื่อไล่ตาม gradient ไปยังชุดมุมที่ดีที่สุดในบริเวณใกล้เคียง ในกรณีของ computational fluid dynamics (CFD) ลำดับการ optimize เริ่มต้นให้ผลลัพธ์ที่ดีที่สุด — แต่ในกรณีของ material deformation (MD) แม้ค่าเริ่มต้นจะให้ผลดี คุณก็กำหนดค่าเพิ่มเติมเพื่อประโยชน์เฉพาะปัญหาได้

หมายเหตุ สำหรับตัวแปรแต่ละตัวของฟังก์ชัน เราระบุจำนวน qubits (ซึ่งคุณปรับเปลี่ยนได้) โดยการซ้อน 10 Circuit เหมือนกันและประเมิน 10 observable เหมือนกันบน qubits ที่ต่างกันตลอด Circuit ใหญ่หนึ่ง Circuit คุณลด noise ภายในกระบวนการ CMA optimization ได้ โดยอาศัยวิธี noise learner และลดจำนวน shots ที่ต้องการอย่างมีนัยสำคัญ
## Input/output
### Computational Fluid Dynamics
สมการ Burgers' แบบ inviscid สร้างแบบจำลองของไหลไม่มีความหนืดที่ไหลดังนี้:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ แทนฟิลด์ความเร็วของของไหล กรณีการใช้งานนี้มีเงื่อนไขขอบเขตทางเวลา: คุณเลือกเงื่อนไขเริ่มต้นและปล่อยให้ระบบผ่อนคลาย ปัจจุบันเงื่อนไขเริ่มต้นที่ยอมรับได้มีเพียงฟังก์ชันเชิงเส้น: $ax + b$

อาร์กิวเมนต์สำหรับสมการอนุพันธ์ของ CFD อยู่บน grid คงที่ ดังนี้:

- $t$ อยู่ระหว่าง 0 ถึง 0.95 พร้อม 30 จุดตัวอย่าง $x$ อยู่ระหว่าง 0 ถึง 0.95 พร้อม step size 0.2375

### Material Deformation
กรณีการใช้งานนี้เน้นที่การเสียรูปแบบ hypoelastic ด้วยการทดสอบแรงดึงแบบ 1 มิติ ซึ่งแท่งที่ตรึงอยู่ในพื้นที่ถูกดึงที่ปลายอีกด้าน เราอธิบายปัญหาดังนี้:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ แทน bulk modulus ของวัสดุที่ถูกยืด $n$ ตัวชี้กำลังของ power law $b$ แรงต่อมวลหน่วย $\epsilon_0$ ขีดจำกัด proportional stress $\sigma_0$ ขีดจำกัด proportional strain $u$ ฟังก์ชัน stress และ $\sigma$ ฟังก์ชัน strain

แท่งที่พิจารณามีความยาวหนึ่งหน่วย กรณีการใช้งานนี้มีเงื่อนไขขอบเขตสำหรับ surface stress $t$ หรือปริมาณงานที่ต้องการเพื่อยืดแท่ง

อาร์กิวเมนต์สำหรับสมการอนุพันธ์ของ MD อยู่บน grid คงที่ ดังนี้:

- $x$ อยู่ระหว่าง 0 ถึง 1 พร้อม step size 0.04
### Input
เพื่อรัน QUICK-PDE Qiskit Function คุณปรับพารามิเตอร์ต่อไปนี้ได้:

| ชื่อ              | ประเภท                                               | คำอธิบาย                                                                                                                                                                                                                                                                         | เฉพาะ Use-case | ตัวอย่าง                 |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------- | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | สลับเพื่อเลือกระบบสมการอนุพันธ์ที่ต้องการแก้                                                                                                                                                                                                                                   | ไม่               | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | พารามิเตอร์ของสมการอนุพันธ์ (ดูตารางถัดไปสำหรับรายละเอียด)                                                                                                                                                                                                                    | ไม่               | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | จำนวน qubits ต่อฟังก์ชันและต่อตัวแปร ค่าที่ optimize แล้วถูกเลือกโดยฟังก์ชัน แต่ถ้าต้องการลองหาการผสมที่ดีกว่า สามารถแทนที่ค่าเริ่มต้นได้                                                                                                                                   | ไม่               | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | ความลึกของ ansatz ต่อฟังก์ชัน ค่าที่ optimize แล้วถูกเลือกโดยฟังก์ชัน แต่ถ้าต้องการลองหาการผสมที่ดีกว่า สามารถแทนที่ค่าเริ่มต้นได้                                                                                                                                          | ไม่               | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | Optimizers ที่ใช้ ได้แก่ "CMAES" จาก [`cma` python library](https://github.com/CMA-ES/pycma) หรือหนึ่งใน [optimizers ของ scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)                                                              | MD                | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | จำนวน shots ที่ใช้รัน Circuit แต่ละ Circuit เนื่องจากต้องการขั้นตอนการ optimize หลายขั้น ความยาวของรายการต้องเท่ากับจำนวน optimizer ที่ใช้ (4 สำหรับ CFD) ค่าเริ่มต้นคือ `[50_000] * nb_optimizers` สำหรับ MD และ `[5_00, 2_000, 5_000, 10_000]` สำหรับ CFD | ไม่               | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | ตัวเลือกที่ส่งให้ optimizer รายละเอียดของ input นี้ขึ้นอยู่กับ optimizer ที่ใช้ สำหรับรายละเอียด ดูเอกสารของ optimizer ที่ใช้                                                                                                                                               | MD                | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | เลือกว่าจะเริ่มต้นด้วยมุมสุ่มหรือมุมที่เลือกอย่างชาญฉลาด โปรดทราบว่ามุมที่เลือกอย่างชาญฉลาดอาจไม่ทำงานในทุกกรณี ค่าเริ่มต้นคือ `"PHYSICALLY_INFORMED"`                                                                                                                       | ไม่               | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | ชื่อ Backend ที่ต้องการใช้                                                                                                                                                                                                                                                       | ไม่               | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | โหมดการ execute ที่ใช้ ค่าเริ่มต้นคือ `"job"`                                                                                                                                                                                                                                   | ไม่               | `"job"`                  |

พารามิเตอร์ของสมการอนุพันธ์ (พารามิเตอร์ทางกายภาพและเงื่อนไขขอบเขต) ควรเป็นไปตามรูปแบบที่กำหนด:

| Use case | Key         | ประเภทค่า | คำอธิบาย                              | ตัวอย่าง |
| -------- | ----------- | ---------- | ---------------------------------------- | ------- |
| CFD      | `a`         | `float`    | Coefficient ของค่าเริ่มต้นของ $u$      | `1.0`   |
| CFD      | `b`         | `float`    | Offset ของค่าเริ่มต้นของ $u$           | `1.0`   |
| MD       | `t`         | `float`    | surface stress                           | `12.0`  |
| MD       | `K`         | `float`    | bulk modulus                             | `100.0` |
| MD       | `n`         | `int`      | power law                                | `4.0`   |
| MD       | `b`         | `float`    | force per unit mass                      | `10.0`  |
| MD       | `epsilon_0` | `float`    | proportional stress limit                | `0.1`   |
| MD       | `sigma_0`   | `float`    | proportional strain limit                | `5.0`   |
### Output
Output คือ dictionary ที่มีรายการจุดตัวอย่างพร้อมค่าของฟังก์ชันที่แต่ละจุดเหล่านี้:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

รูปร่างของ solution array ขึ้นอยู่กับตัวอย่างของตัวแปร:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

การ mapping ระหว่างจุดตัวอย่างของตัวแปรฟังก์ชันกับมิติของ solution array ทำโดยเรียงลำดับตัวอักษรและตัวเลขของชื่อตัวแปร ตัวอย่างเช่น ถ้าตัวแปรคือ `"t"` และ `"x"` แถวของ `solution["functions"]["u"]` แทนค่าของโซลูชันสำหรับ `"t"` คงที่ และคอลัมน์ของ `solution["functions"]["u"]` แทนค่าของโซลูชันสำหรับ `"x"` คงที่

ต่อไปนี้เป็นตัวอย่างวิธีรับค่าของฟังก์ชันสำหรับชุดพิกัดเฉพาะ:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Benchmarks

ตารางต่อไปนี้แสดงสถิติจากการรันฟังก์ชันของเราในหลายครั้ง

| ตัวอย่าง                     | จำนวน qubits | Initialization        | Error     | เวลารวม (นาที) | Runtime usage (นาที) |
| ---------------------------- | ---------------- | --------------------- | --------- | ---------------- | ------------------- |
| สมการ Burgers' แบบ inviscid  | 50               | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66               | 25                  |
| Hypoelastic 1D tensile test  | 18               | `RANDOM`              | $10^{-2}$ | 123              | 100                 |

## เริ่มต้นใช้งาน

กรอก [แบบฟอร์มเพื่อขอเข้าถึงฟังก์ชัน QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK) จากนั้น สมมติว่า [บันทึกบัญชีของคุณแล้ว](/guides/functions#install-qiskit-functions-catalog-client) ไว้ใน local environment ให้เลือกฟังก์ชันดังนี้:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## ตัวอย่าง

ในการเริ่มต้น ลองทำหนึ่งในตัวอย่างต่อไปนี้:

### Computational Fluid Dynamics (CFD)

เมื่อเงื่อนไขเริ่มต้นตั้งเป็น $u(0,x) = x$ ผลลัพธ์มีดังนี้:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

ตรวจสอบ [status](/guides/functions#check-job-status) ของ Qiskit Function workload หรือดึง [ผลลัพธ์](/guides/functions#retrieve-results) ดังนี้:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Material Deformation
กรณีการใช้งาน material deformation ต้องการพารามิเตอร์ทางกายภาพของวัสดุและแรงที่ใช้ ดังนี้:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## ดึงข้อความ error

ถ้าสถานะ workload เป็น `ERROR` ให้ใช้ `job.error_message()` เพื่อดึงข้อความ error มาช่วย debug ดังนี้: